# RadImageNet-VQA unique-image audit

Count the images represented by the original 500 evaluation questions. This notebook includes the saved sample indices and SHA-256 question fingerprints from the project. It reproduces the original seed-42 sampling and validates the dataset rows against those records.

## Kaggle setup

1. Enable Internet. CPU is sufficient.
2. Request/accept access to `raidium/RadImageNet-VQA` on Hugging Face using your account.
3. Add a Kaggle secret named `HF_TOKEN` with read access, and enable it for this notebook.
4. Run all cells. Download the aggregate CSV, JSON, and text report from `/kaggle/working/radimagenet_image_audit`.

Dataset images and question text are used in memory. Outputs contain aggregate counts and provenance. The notebook reports distinct dataset image identifiers when available and independently counts distinct decoded image contents. Identical content can occur under different identifiers, so both counts are retained.


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'datasets', 'huggingface_hub', 'pandas', 'numpy', 'pillow', 'tqdm'])


In [ ]:
import hashlib, io, json, platform
from collections import Counter
from pathlib import Path
import importlib.metadata
import numpy as np
import pandas as pd
from datasets import load_dataset
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient
from tqdm.auto import tqdm

DATASET = 'raidium/RadImageNet-VQA'
SEED = 42
# Set a historical dataset commit here if you have it; otherwise the resolved current commit is recorded.
DATASET_REVISION = None
# Optional explicit column: e.g. 'image_id' or 'metadata.image_id'.
IMAGE_ID_FIELD = None
OUTPUT = Path('/kaggle/working/radimagenet_image_audit')
TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
revision = HfApi().dataset_info(DATASET, revision=DATASET_REVISION, token=TOKEN).sha
print('Dataset commit:', revision)
ds = load_dataset(DATASET, 'benchmark', split='test', revision=revision, token=TOKEN)
print('Dataset rows:', len(ds))
print('Columns:', ds.column_names)
assert len(ds) == 9000, 'Expected 9,000 benchmark rows; investigate dataset revision.'


In [ ]:
SAVED_SAMPLE_CSV = 'idx,question_id,group\n22,abnormal_closed,abnormality_yesno\n82,anatomy_closed_pos,anatomy_yesno\n93,anatomy_mc,anatomy_mcq\n98,pathology_mc,pathology_mcq\n169,pathology_closed_neg,pathology_yesno\n210,anatomy_mc,anatomy_mcq\n215,pathology_mc,pathology_mcq\n222,pathology_closed_pos,pathology_yesno\n262,anatomy_closed_pos,anatomy_yesno\n273,anatomy_mc,anatomy_mcq\n278,pathology_mc,pathology_mcq\n310,abnormal_closed,abnormality_yesno\n330,pathology_closed_pos,pathology_yesno\n331,pathology_closed_neg,pathology_yesno\n340,pathology_closed_neg,pathology_yesno\n346,abnormal_closed,abnormality_yesno\n354,anatomy_mc,anatomy_mcq\n359,pathology_mc,pathology_mcq\n379,anatomy_closed_pos,anatomy_yesno\n382,abnormal_closed,abnormality_yesno\n384,pathology_closed_pos,pathology_yesno\n393,pathology_closed_pos,pathology_yesno\n411,pathology_closed_pos,pathology_yesno\n424,anatomy_closed_pos,anatomy_yesno\n484,pathology_closed_neg,pathology_yesno\n489,anatomy_mc,anatomy_mcq\n494,pathology_mc,pathology_mcq\n505,anatomy_closed_pos,anatomy_yesno\n533,anatomy_closed_neg,anatomy_yesno\n534,anatomy_mc,anatomy_mcq\n539,pathology_mc,pathology_mcq\n555,pathology_closed_pos,pathology_yesno\n570,anatomy_mc,anatomy_mcq\n575,pathology_mc,pathology_mcq\n595,anatomy_closed_pos,anatomy_yesno\n597,anatomy_mc,anatomy_mcq\n602,pathology_mc,pathology_mcq\n606,anatomy_mc,anatomy_mcq\n611,pathology_mc,pathology_mcq\n631,anatomy_closed_pos,anatomy_yesno\n633,anatomy_mc,anatomy_mcq\n638,pathology_mc,pathology_mcq\n686,anatomy_closed_neg,anatomy_yesno\n687,anatomy_mc,anatomy_mcq\n692,pathology_mc,pathology_mcq\n715,abnormal_closed,abnormality_yesno\n762,pathology_closed_pos,pathology_yesno\n778,abnormal_closed,abnormality_yesno\n795,anatomy_mc,anatomy_mcq\n798,pathology_closed_pos,pathology_yesno\n800,pathology_mc,pathology_mcq\n811,anatomy_closed_pos,anatomy_yesno\n825,pathology_closed_pos,pathology_yesno\n832,abnormal_closed,abnormality_yesno\n838,anatomy_closed_pos,anatomy_yesno\n859,abnormal_closed,abnormality_yesno\n861,pathology_closed_pos,pathology_yesno\n866,anatomy_closed_neg,anatomy_yesno\n867,anatomy_mc,anatomy_mcq\n872,pathology_mc,pathology_mcq\n901,anatomy_closed_pos,anatomy_yesno\n911,anatomy_closed_neg,anatomy_yesno\n912,anatomy_mc,anatomy_mcq\n917,pathology_mc,pathology_mcq\n922,abnormal_closed,abnormality_yesno\n925,pathology_closed_neg,pathology_yesno\n966,anatomy_mc,anatomy_mcq\n971,pathology_mc,pathology_mcq\n987,pathology_closed_pos,pathology_yesno\n1014,pathology_closed_pos,pathology_yesno\n1048,abnormal_closed,abnormality_yesno\n1057,abnormal_closed,abnormality_yesno\n1120,abnormal_closed,abnormality_yesno\n1183,abnormal_closed,abnormality_yesno\n1226,anatomy_closed_neg,anatomy_yesno\n1227,anatomy_mc,anatomy_mcq\n1228,abnormal_closed,abnormality_yesno\n1232,pathology_mc,pathology_mcq\n1253,anatomy_closed_neg,anatomy_yesno\n1254,anatomy_mc,anatomy_mcq\n1259,pathology_mc,pathology_mcq\n1291,abnormal_closed,abnormality_yesno\n1302,pathology_closed_pos,pathology_yesno\n1347,pathology_closed_pos,pathology_yesno\n1462,abnormal_closed,abnormality_yesno\n1507,abnormal_closed,abnormality_yesno\n1516,abnormal_closed,abnormality_yesno\n1549,anatomy_closed_pos,anatomy_yesno\n1568,anatomy_closed_neg,anatomy_yesno\n1569,anatomy_mc,anatomy_mcq\n1574,pathology_mc,pathology_mcq\n1608,pathology_closed_pos,pathology_yesno\n1681,pathology_closed_neg,pathology_yesno\n1690,pathology_closed_neg,pathology_yesno\n1784,anatomy_closed_neg,anatomy_yesno\n1785,anatomy_mc,anatomy_mcq\n1786,abnormal_closed,abnormality_yesno\n1790,pathology_mc,pathology_mcq\n1806,pathology_closed_pos,pathology_yesno\n1843,pathology_closed_neg,pathology_yesno\n1884,anatomy_mc,anatomy_mcq\n1885,abnormal_closed,abnormality_yesno\n1889,pathology_mc,pathology_mcq\n1892,anatomy_closed_neg,anatomy_yesno\n1893,anatomy_mc,anatomy_mcq\n1898,pathology_mc,pathology_mcq\n1918,anatomy_closed_pos,anatomy_yesno\n1939,abnormal_closed,abnormality_yesno\n1965,anatomy_mc,anatomy_mcq\n1970,pathology_mc,pathology_mcq\n1984,abnormal_closed,abnormality_yesno\n1987,pathology_closed_neg,pathology_yesno\n1992,anatomy_mc,anatomy_mcq\n1997,pathology_mc,pathology_mcq\n2095,pathology_closed_neg,pathology_yesno\n2098,anatomy_closed_pos,anatomy_yesno\n2117,anatomy_closed_neg,anatomy_yesno\n2118,anatomy_mc,anatomy_mcq\n2123,pathology_mc,pathology_mcq\n2146,abnormal_closed,abnormality_yesno\n2182,abnormal_closed,abnormality_yesno\n2283,pathology_closed_pos,pathology_yesno\n2287,anatomy_closed_pos,anatomy_yesno\n2299,abnormal_closed,abnormality_yesno\n2346,pathology_closed_pos,pathology_yesno\n2352,anatomy_mc,anatomy_mcq\n2357,pathology_mc,pathology_mcq\n2371,abnormal_closed,abnormality_yesno\n2377,anatomy_closed_pos,anatomy_yesno\n2409,pathology_closed_pos,pathology_yesno\n2434,abnormal_closed,abnormality_yesno\n2478,anatomy_mc,anatomy_mcq\n2483,pathology_mc,pathology_mcq\n2495,anatomy_closed_neg,anatomy_yesno\n2496,anatomy_mc,anatomy_mcq\n2501,pathology_mc,pathology_mcq\n2522,anatomy_closed_neg,anatomy_yesno\n2523,anatomy_mc,anatomy_mcq\n2528,pathology_mc,pathology_mcq\n2533,abnormal_closed,abnormality_yesno\n2560,abnormal_closed,abnormality_yesno\n2587,abnormal_closed,abnormality_yesno\n2590,pathology_closed_neg,pathology_yesno\n2604,anatomy_mc,anatomy_mcq\n2609,pathology_mc,pathology_mcq\n2616,pathology_closed_pos,pathology_yesno\n2631,anatomy_mc,anatomy_mcq\n2632,abnormal_closed,abnormality_yesno\n2636,pathology_mc,pathology_mcq\n2638,anatomy_closed_pos,anatomy_yesno\n2661,pathology_closed_pos,pathology_yesno\n2667,anatomy_mc,anatomy_mcq\n2670,pathology_closed_pos,pathology_yesno\n2672,pathology_mc,pathology_mcq\n2693,anatomy_closed_neg,anatomy_yesno\n2694,anatomy_mc,anatomy_mcq\n2699,pathology_mc,pathology_mcq\n2719,anatomy_closed_pos,anatomy_yesno\n2766,anatomy_mc,anatomy_mcq\n2771,pathology_mc,pathology_mcq\n2787,pathology_closed_pos,pathology_yesno\n2794,abnormal_closed,abnormality_yesno\n2810,anatomy_closed_neg,anatomy_yesno\n2811,anatomy_mc,anatomy_mcq\n2816,pathology_mc,pathology_mcq\n2827,anatomy_closed_pos,anatomy_yesno\n2859,pathology_closed_pos,pathology_yesno\n2865,anatomy_mc,anatomy_mcq\n2866,abnormal_closed,abnormality_yesno\n2870,pathology_mc,pathology_mcq\n2873,anatomy_closed_neg,anatomy_yesno\n2874,anatomy_mc,anatomy_mcq\n2879,pathology_mc,pathology_mcq\n2893,abnormal_closed,abnormality_yesno\n2938,abnormal_closed,abnormality_yesno\n2946,anatomy_mc,anatomy_mcq\n2951,pathology_mc,pathology_mcq\n2954,anatomy_closed_neg,anatomy_yesno\n2955,anatomy_mc,anatomy_mcq\n2959,pathology_closed_neg,pathology_yesno\n2960,pathology_mc,pathology_mcq\n2977,pathology_closed_neg,pathology_yesno\n2994,pathology_closed_pos,pathology_yesno\n3028,abnormal_closed,abnormality_yesno\n3037,abnormal_closed,abnormality_yesno\n3094,pathology_closed_neg,pathology_yesno\n3116,anatomy_closed_neg,anatomy_yesno\n3117,anatomy_mc,anatomy_mcq\n3120,pathology_closed_pos,pathology_yesno\n3122,pathology_mc,pathology_mcq\n3235,abnormal_closed,abnormality_yesno\n3246,pathology_closed_pos,pathology_yesno\n3250,anatomy_closed_pos,anatomy_yesno\n3332,anatomy_closed_neg,anatomy_yesno\n3333,anatomy_mc,anatomy_mcq\n3338,pathology_mc,pathology_mcq\n3341,anatomy_closed_neg,anatomy_yesno\n3342,anatomy_mc,anatomy_mcq\n3347,pathology_mc,pathology_mcq\n3417,pathology_closed_pos,pathology_yesno\n3441,anatomy_mc,anatomy_mcq\n3446,pathology_mc,pathology_mcq\n3505,abnormal_closed,abnormality_yesno\n3532,abnormal_closed,abnormality_yesno\n3538,anatomy_closed_pos,anatomy_yesno\n3574,anatomy_closed_pos,anatomy_yesno\n3624,pathology_closed_pos,pathology_yesno\n3700,anatomy_closed_pos,anatomy_yesno\n3701,anatomy_closed_neg,anatomy_yesno\n3702,anatomy_mc,anatomy_mcq\n3707,pathology_mc,pathology_mcq\n3739,abnormal_closed,abnormality_yesno\n3768,pathology_closed_pos,pathology_yesno\n3781,anatomy_closed_pos,anatomy_yesno\n3822,pathology_closed_pos,pathology_yesno\n3838,abnormal_closed,abnormality_yesno\n3858,pathology_closed_pos,pathology_yesno\n3867,pathology_closed_pos,pathology_yesno\n3927,anatomy_mc,anatomy_mcq\n3932,pathology_mc,pathology_mcq\n3953,anatomy_closed_neg,anatomy_yesno\n3954,anatomy_mc,anatomy_mcq\n3955,abnormal_closed,abnormality_yesno\n3959,pathology_mc,pathology_mcq\n3973,abnormal_closed,abnormality_yesno\n3997,anatomy_closed_pos,anatomy_yesno\n4020,pathology_closed_pos,pathology_yesno\n4036,abnormal_closed,abnormality_yesno\n4062,anatomy_mc,anatomy_mcq\n4067,pathology_mc,pathology_mcq\n4093,pathology_closed_neg,pathology_yesno\n4105,anatomy_closed_pos,anatomy_yesno\n4108,abnormal_closed,abnormality_yesno\n4204,anatomy_closed_pos,anatomy_yesno\n4222,anatomy_closed_pos,anatomy_yesno\n4254,pathology_closed_pos,pathology_yesno\n4270,abnormal_closed,abnormality_yesno\n4282,pathology_closed_neg,pathology_yesno\n4351,abnormal_closed,abnormality_yesno\n4363,pathology_closed_neg,pathology_yesno\n4366,anatomy_closed_pos,anatomy_yesno\n4396,abnormal_closed,abnormality_yesno\n4440,anatomy_mc,anatomy_mcq\n4445,pathology_mc,pathology_mcq\n4470,pathology_closed_pos,pathology_yesno\n4493,anatomy_closed_neg,anatomy_yesno\n4494,anatomy_mc,anatomy_mcq\n4499,pathology_mc,pathology_mcq\n4522,abnormal_closed,abnormality_yesno\n4552,pathology_closed_neg,pathology_yesno\n4588,pathology_closed_neg,pathology_yesno\n4619,anatomy_closed_neg,anatomy_yesno\n4620,anatomy_mc,anatomy_mcq\n4625,pathology_mc,pathology_mcq\n4672,anatomy_closed_pos,anatomy_yesno\n4691,anatomy_closed_neg,anatomy_yesno\n4692,anatomy_mc,anatomy_mcq\n4697,pathology_mc,pathology_mcq\n4701,anatomy_mc,anatomy_mcq\n4706,pathology_mc,pathology_mcq\n4741,pathology_closed_neg,pathology_yesno\n4745,anatomy_closed_neg,anatomy_yesno\n4746,anatomy_mc,anatomy_mcq\n4751,pathology_mc,pathology_mcq\n4758,pathology_closed_pos,pathology_yesno\n4776,pathology_closed_pos,pathology_yesno\n4789,anatomy_closed_pos,anatomy_yesno\n4792,abnormal_closed,abnormality_yesno\n4801,abnormal_closed,abnormality_yesno\n4812,pathology_closed_pos,pathology_yesno\n4819,abnormal_closed,abnormality_yesno\n4828,abnormal_closed,abnormality_yesno\n4837,abnormal_closed,abnormality_yesno\n4846,abnormal_closed,abnormality_yesno\n4881,anatomy_mc,anatomy_mcq\n4886,pathology_mc,pathology_mcq\n4890,anatomy_mc,anatomy_mcq\n4893,pathology_closed_pos,pathology_yesno\n4895,pathology_mc,pathology_mcq\n4927,abnormal_closed,abnormality_yesno\n4935,anatomy_mc,anatomy_mcq\n4940,pathology_mc,pathology_mcq\n4943,anatomy_closed_neg,anatomy_yesno\n4944,anatomy_mc,anatomy_mcq\n4949,pathology_mc,pathology_mcq\n4966,pathology_closed_neg,pathology_yesno\n5019,pathology_closed_pos,pathology_yesno\n5020,pathology_closed_neg,pathology_yesno\n5059,anatomy_closed_pos,anatomy_yesno\n5074,pathology_closed_neg,pathology_yesno\n5110,pathology_closed_neg,pathology_yesno\n5133,anatomy_mc,anatomy_mcq\n5138,pathology_mc,pathology_mcq\n5181,pathology_closed_pos,pathology_yesno\n5188,abnormal_closed,abnormality_yesno\n5199,pathology_closed_pos,pathology_yesno\n5205,anatomy_mc,anatomy_mcq\n5206,abnormal_closed,abnormality_yesno\n5210,pathology_mc,pathology_mcq\n5258,anatomy_closed_neg,anatomy_yesno\n5259,anatomy_mc,anatomy_mcq\n5264,pathology_mc,pathology_mcq\n5296,abnormal_closed,abnormality_yesno\n5304,anatomy_mc,anatomy_mcq\n5305,abnormal_closed,abnormality_yesno\n5309,pathology_mc,pathology_mcq\n5313,anatomy_mc,anatomy_mcq\n5318,pathology_mc,pathology_mcq\n5411,anatomy_closed_neg,anatomy_yesno\n5412,anatomy_mc,anatomy_mcq\n5416,pathology_closed_neg,pathology_yesno\n5417,pathology_mc,pathology_mcq\n5431,abnormal_closed,abnormality_yesno\n5439,anatomy_mc,anatomy_mcq\n5444,pathology_mc,pathology_mcq\n5451,pathology_closed_pos,pathology_yesno\n5479,pathology_closed_neg,pathology_yesno\n5556,anatomy_mc,anatomy_mcq\n5561,pathology_mc,pathology_mcq\n5591,anatomy_closed_neg,anatomy_yesno\n5592,anatomy_mc,anatomy_mcq\n5597,pathology_mc,pathology_mcq\n5636,anatomy_closed_neg,anatomy_yesno\n5637,anatomy_mc,anatomy_mcq\n5642,pathology_mc,pathology_mcq\n5647,abnormal_closed,abnormality_yesno\n5658,pathology_closed_pos,pathology_yesno\n5676,pathology_closed_pos,pathology_yesno\n5698,anatomy_closed_pos,anatomy_yesno\n5726,anatomy_closed_neg,anatomy_yesno\n5727,anatomy_mc,anatomy_mcq\n5732,pathology_mc,pathology_mcq\n5764,abnormal_closed,abnormality_yesno\n5836,abnormal_closed,abnormality_yesno\n5871,anatomy_mc,anatomy_mcq\n5876,pathology_mc,pathology_mcq\n5942,anatomy_closed_neg,anatomy_yesno\n5943,anatomy_mc,anatomy_mcq\n5944,abnormal_closed,abnormality_yesno\n5948,pathology_mc,pathology_mcq\n5961,anatomy_mc,anatomy_mcq\n5966,pathology_mc,pathology_mcq\n5971,abnormal_closed,abnormality_yesno\n6034,abnormal_closed,abnormality_yesno\n6036,pathology_closed_pos,pathology_yesno\n6104,anatomy_closed_neg,anatomy_yesno\n6105,anatomy_mc,anatomy_mcq\n6110,pathology_mc,pathology_mcq\n6114,anatomy_mc,anatomy_mcq\n6119,pathology_mc,pathology_mcq\n6133,abnormal_closed,abnormality_yesno\n6166,anatomy_closed_pos,anatomy_yesno\n6185,anatomy_closed_neg,anatomy_yesno\n6186,anatomy_mc,anatomy_mcq\n6191,pathology_mc,pathology_mcq\n6229,anatomy_closed_pos,anatomy_yesno\n6231,anatomy_mc,anatomy_mcq\n6236,pathology_mc,pathology_mcq\n6292,anatomy_closed_pos,anatomy_yesno\n6421,abnormal_closed,abnormality_yesno\n6429,anatomy_mc,anatomy_mcq\n6434,pathology_mc,pathology_mcq\n6562,anatomy_closed_pos,anatomy_yesno\n6585,pathology_closed_pos,pathology_yesno\n6635,anatomy_closed_neg,anatomy_yesno\n6636,anatomy_mc,anatomy_mcq\n6641,pathology_mc,pathology_mcq\n6643,anatomy_closed_pos,anatomy_yesno\n6649,pathology_closed_neg,pathology_yesno\n6652,anatomy_closed_pos,anatomy_yesno\n6662,anatomy_closed_neg,anatomy_yesno\n6663,anatomy_mc,anatomy_mcq\n6668,pathology_mc,pathology_mcq\n6672,anatomy_mc,anatomy_mcq\n6677,pathology_mc,pathology_mcq\n6706,anatomy_closed_pos,anatomy_yesno\n6780,anatomy_mc,anatomy_mcq\n6781,abnormal_closed,abnormality_yesno\n6785,pathology_mc,pathology_mcq\n6817,abnormal_closed,abnormality_yesno\n6844,abnormal_closed,abnormality_yesno\n6851,anatomy_closed_neg,anatomy_yesno\n6852,anatomy_mc,anatomy_mcq\n6857,pathology_mc,pathology_mcq\n6864,pathology_closed_pos,pathology_yesno\n6879,anatomy_mc,anatomy_mcq\n6884,pathology_mc,pathology_mcq\n6910,pathology_closed_neg,pathology_yesno\n6961,abnormal_closed,abnormality_yesno\n6970,abnormal_closed,abnormality_yesno\n6972,pathology_closed_pos,pathology_yesno\n7006,abnormal_closed,abnormality_yesno\n7086,anatomy_mc,anatomy_mcq\n7089,pathology_closed_pos,pathology_yesno\n7091,pathology_mc,pathology_mcq\n7096,abnormal_closed,abnormality_yesno\n7117,pathology_closed_neg,pathology_yesno\n7141,abnormal_closed,abnormality_yesno\n7159,abnormal_closed,abnormality_yesno\n7188,pathology_closed_pos,pathology_yesno\n7195,abnormal_closed,abnormality_yesno\n7207,pathology_closed_neg,pathology_yesno\n7301,anatomy_closed_neg,anatomy_yesno\n7302,anatomy_mc,anatomy_mcq\n7307,pathology_mc,pathology_mcq\n7309,anatomy_closed_pos,anatomy_yesno\n7366,abnormal_closed,abnormality_yesno\n7378,pathology_closed_neg,pathology_yesno\n7383,anatomy_mc,anatomy_mcq\n7388,pathology_mc,pathology_mcq\n7437,anatomy_mc,anatomy_mcq\n7442,pathology_mc,pathology_mcq\n7447,abnormal_closed,abnormality_yesno\n7495,pathology_closed_neg,pathology_yesno\n7519,abnormal_closed,abnormality_yesno\n7535,anatomy_closed_neg,anatomy_yesno\n7536,anatomy_mc,anatomy_mcq\n7539,pathology_closed_pos,pathology_yesno\n7541,pathology_mc,pathology_mcq\n7612,pathology_closed_neg,pathology_yesno\n7648,pathology_closed_neg,pathology_yesno\n7660,anatomy_closed_pos,anatomy_yesno\n7693,pathology_closed_neg,pathology_yesno\n7696,anatomy_closed_pos,anatomy_yesno\n7699,abnormal_closed,abnormality_yesno\n7710,pathology_closed_pos,pathology_yesno\n7714,anatomy_closed_pos,anatomy_yesno\n7733,anatomy_closed_neg,anatomy_yesno\n7734,anatomy_mc,anatomy_mcq\n7739,pathology_mc,pathology_mcq\n7750,anatomy_closed_pos,anatomy_yesno\n7797,anatomy_mc,anatomy_mcq\n7802,pathology_mc,pathology_mcq\n7807,abnormal_closed,abnormality_yesno\n7825,abnormal_closed,abnormality_yesno\n7831,anatomy_closed_pos,anatomy_yesno\n7890,pathology_closed_pos,pathology_yesno\n7899,pathology_closed_pos,pathology_yesno\n7914,anatomy_mc,anatomy_mcq\n7919,pathology_mc,pathology_mcq\n7949,anatomy_closed_neg,anatomy_yesno\n7950,anatomy_mc,anatomy_mcq\n7953,pathology_closed_pos,pathology_yesno\n7955,pathology_mc,pathology_mcq\n8005,abnormal_closed,abnormality_yesno\n8023,abnormal_closed,abnormality_yesno\n8062,pathology_closed_neg,pathology_yesno\n8093,anatomy_closed_neg,anatomy_yesno\n8094,anatomy_mc,anatomy_mcq\n8099,pathology_mc,pathology_mcq\n8111,anatomy_closed_neg,anatomy_yesno\n8112,anatomy_mc,anatomy_mcq\n8117,pathology_mc,pathology_mcq\n8120,anatomy_closed_neg,anatomy_yesno\n8121,anatomy_mc,anatomy_mcq\n8126,pathology_mc,pathology_mcq\n8131,abnormal_closed,abnormality_yesno\n8167,abnormal_closed,abnormality_yesno\n8215,pathology_closed_neg,pathology_yesno\n8290,anatomy_closed_pos,anatomy_yesno\n8293,abnormal_closed,abnormality_yesno\n8317,anatomy_closed_pos,anatomy_yesno\n8318,anatomy_closed_neg,anatomy_yesno\n8319,anatomy_mc,anatomy_mcq\n8324,pathology_mc,pathology_mcq\n8365,abnormal_closed,abnormality_yesno\n8380,anatomy_closed_pos,anatomy_yesno\n8437,abnormal_closed,abnormality_yesno\n8444,anatomy_closed_neg,anatomy_yesno\n8445,anatomy_mc,anatomy_mcq\n8450,pathology_mc,pathology_mcq\n8455,abnormal_closed,abnormality_yesno\n8509,abnormal_closed,abnormality_yesno\n8525,anatomy_closed_neg,anatomy_yesno\n8526,anatomy_mc,anatomy_mcq\n8531,pathology_mc,pathology_mcq\n8563,abnormal_closed,abnormality_yesno\n8593,pathology_closed_neg,pathology_yesno\n8599,abnormal_closed,abnormality_yesno\n8626,abnormal_closed,abnormality_yesno\n8628,pathology_closed_pos,pathology_yesno\n8664,pathology_closed_pos,pathology_yesno\n8704,anatomy_closed_pos,anatomy_yesno\n8737,pathology_closed_neg,pathology_yesno\n8759,anatomy_closed_neg,anatomy_yesno\n8760,anatomy_mc,anatomy_mcq\n8765,pathology_mc,pathology_mcq\n8768,anatomy_closed_neg,anatomy_yesno\n8769,anatomy_mc,anatomy_mcq\n8774,pathology_mc,pathology_mcq\n8848,anatomy_closed_pos,anatomy_yesno\n8876,anatomy_closed_neg,anatomy_yesno\n8877,anatomy_mc,anatomy_mcq\n8878,abnormal_closed,abnormality_yesno\n8882,pathology_mc,pathology_mcq\n8889,pathology_closed_pos,pathology_yesno\n8959,abnormal_closed,abnormality_yesno\n8961,pathology_closed_pos,pathology_yesno\n8985,anatomy_mc,anatomy_mcq\n8990,pathology_mc,pathology_mcq\n'
QUESTION_SHA256 = {'22': '70dba65866edff0ab960e98994ae7b784c94fe5670b68b5b8568a06bb10d08cf', '82': 'f1fbbdd03a1a041e9a85abc83d788e8617ccdb64817df0b15e26c3ec1bc337b0', '93': '142758aff462348a4a4b9a8d6afb4e61c4ac4b6a71ccdbce6855192ce0845895', '98': 'f9678ba9e852139905d532f36764bf01da9442302322dc859fdec0413762e855', '169': '84df8888d34f90a3529b11720b24c569a470d5db15eba40d2ca3eb66399e4d96', '210': '142758aff462348a4a4b9a8d6afb4e61c4ac4b6a71ccdbce6855192ce0845895', '215': 'cdb9d08f8d29b32ab5384800aee61346b317c5343582918e57206c4abc79955a', '222': '2d2cb6ae1bdbc8bcf7da490fb3776765a830c97f150a57f598449e20015ea124', '262': 'bb0f6c2116adf12144986d41ad99e13f42250b4b44d745fc30ef94c2506068c1', '273': '83f53b49c08848bcc25b31738ac9d1a48cfb9aa0bf44bef277d40226f0a233c5', '278': '7ee215fd727ced2f056600f84cfb3e901f64b39fe309aae820281838f68e33a7', '310': 'fdd5306bebb0ff517de97a6f29da43c837ffe1ac40f8f1f9d7326e98a9d357f7', '330': '8b39afad5da664238199853d0205d1d802a84eec6fb2123057b51016ad9670c3', '331': 'a0a5d4cbfe4ab7aa1ec623b7f6481e5840e316c182171b8cefbd36962bba5ff3', '340': 'ce8b46bb4f6f2fc21d454106e1a3d870ae38eb914d2cf5c356cd146e26f89ee4', '346': '70dba65866edff0ab960e98994ae7b784c94fe5670b68b5b8568a06bb10d08cf', '354': '9e60b030e3fc3416bad089537b6e1b323d5ef6d06e9f66546b63d156fd4ffa45', '359': '7ee215fd727ced2f056600f84cfb3e901f64b39fe309aae820281838f68e33a7', '379': '0b2289a5b1117dcbdaedaab8311f802d09aa3e931cf3d95b3bc0becad7e29431', '382': '70dba65866edff0ab960e98994ae7b784c94fe5670b68b5b8568a06bb10d08cf', '384': '0c0c73c05b39607890e37759314a3bfb1927f9533cdd9294a127517d5186d862', '393': '2b46a1d4003e11a1fcd57e00c4107efc6d92fa98457b11b6cb60797f14ef57f5', '411': '2dc39d9e385ddf99a7f11a2a55be83971fd2d45ed9e6090a9f7d063d73d8c129', '424': 'f31ccacd56cffd006b148b9d40ed35bd8c5290a1c056fed2c9aca93e0729ad78', '484': '147762ea6dd2aed8dd882427a81538eccbaeba9b3845416981595cbc228c7666', '489': '9e60b030e3fc3416bad089537b6e1b323d5ef6d06e9f66546b63d156fd4ffa45', '494': '6c28bfe3dab838da93f7cefaee0f456c859283d9a5238a2910728f7cfd7a1c8a', '505': '4ef9d485257a7e052b8731e0e7cecd2d1acfd13f5d4e9f3ed9131f5cbd2dd15f', '533': '6f9d5c5a89c44bffad714d20b0a432b99388cdb66836c54d118172630ab07223', '534': '9e60b030e3fc3416bad089537b6e1b323d5ef6d06e9f66546b63d156fd4ffa45', '539': 'f9678ba9e852139905d532f36764bf01da9442302322dc859fdec0413762e855', '555': '79d34e8c24c0130a66512ef14c6009b32c65f52f447be2dd4b0463b936a3b468', '570': '83f53b49c08848bcc25b31738ac9d1a48cfb9aa0bf44bef277d40226f0a233c5', '575': '3dcb43a7d32fa6f1c227100d15b315db8519fcef7878df9dee34f79c39d84d1d', '595': '1a019a64a45e235c260637243053c31a84369338428d8bd4c58822942a7490e6', '597': '9e60b030e3fc3416bad089537b6e1b323d5ef6d06e9f66546b63d156fd4ffa45', '602': '3dcb43a7d32fa6f1c227100d15b315db8519fcef7878df9dee34f79c39d84d1d', '606': '83f53b49c08848bcc25b31738ac9d1a48cfb9aa0bf44bef277d40226f0a233c5', '611': 'cdb9d08f8d29b32ab5384800aee61346b317c5343582918e57206c4abc79955a', '631': 'b43d75e58feb5bec6af55bf31e3f8df270870b0ff26cc19aeb84de4bd2825d5d', '633': '83f53b49c08848bcc25b31738ac9d1a48cfb9aa0bf44bef277d40226f0a233c5', '638': 'f9678ba9e852139905d532f36764bf01da9442302322dc859fdec0413762e855', '686': '12cb8533bb086cbc6b9e99163dad6c8ccc8361564c8ec2005262580fb48a683b', '687': '142758aff462348a4a4b9a8d6afb4e61c4ac4b6a71ccdbce6855192ce0845895', '692': '17a0916a59b078ec4dafecb6804e0a74a7c224cde4ccdbe91a47f0b50d863431', '715': '70dba65866edff0ab960e98994ae7b784c94fe5670b68b5b8568a06bb10d08cf', '762': '4d658e8f590d8de4e6a4507e5ea72455485de54f556bbbcaecc75bf8191c12ed', '778': '1297c63eae1002008ad50849f13685208906448bd4157b56facb3e9b84130ec3', '795': '142758aff462348a4a4b9a8d6afb4e61c4ac4b6a71ccdbce6855192ce0845895', '798': '81032ec5a734e997caf61050e999050e7d9d0f630bce032a53664a7b0d989746', '800': 'f9678ba9e852139905d532f36764bf01da9442302322dc859fdec0413762e855', '811': '174360acea5bfdfb12c1c73c88e5609360b0ba2278c5dc5e293f3c7d8401bed2', '825': '05d9d3924c6c595676fdf89014ff559f08b4d9890988ad7cd7c1ba1cc70d5a91', '832': '93bc88d036a697d382446544594c15b2742ab90c25e44517a263621004aeb447', '838': 'c9a7e1fcdd31a68a7e5b8bd830f01663a2194a642df60b87c934cfdcb1241bd1', '859': '93bc88d036a697d382446544594c15b2742ab90c25e44517a263621004aeb447', '861': 'ef3f02ae44347c7492e2803768e497813e8685d27a589620f7c94532e6db6e6b', '866': 'f1fbbdd03a1a041e9a85abc83d788e8617ccdb64817df0b15e26c3ec1bc337b0', '867': '142758aff462348a4a4b9a8d6afb4e61c4ac4b6a71ccdbce6855192ce0845895', '872': 'cdb9d08f8d29b32ab5384800aee61346b317c5343582918e57206c4abc79955a', '901': '2c593b6487a9d04c23d04ce484c496d3d63f55956055ada020e4fb9fcd52a4d1', '911': 'cd70225a5ad47a8ecbf6214f41556d835194dadd34e70d15801528a865d9af14', '912': '83f53b49c08848bcc25b31738ac9d1a48cfb9aa0bf44bef277d40226f0a233c5', '917': 'f9678ba9e852139905d532f36764bf01da9442302322dc859fdec0413762e855', '922': '93bc88d036a697d382446544594c15b2742ab90c25e44517a263621004aeb447', '925': '61bacc7262dbc6f1b4906d5f703c92bd14dda3108c5646b8bf181479f026b009', '966': '9e60b030e3fc3416bad089537b6e1b323d5ef6d06e9f66546b63d156fd4ffa45', '971': 'cdb9d08f8d29b32ab5384800aee61346b317c5343582918e57206c4abc79955a', '987': '016fb6e5bc3f9da803b9c229db796e9de184cbdd29f09a267dd423eeaa3f21fc', '1014': '09955e82969a7492f09dfeeb32b7eca47ca4133cd577c5ff9272bd724b1e7d37', '1048': 'a730cea9827bc512da8df709301c5767a7817dcfc9b71ef55226ee7aa305806e', '1057': 'd2de412a57e1536a7e0935eb65c0416fd7e73014e3581fb0110766a6ff5dcef1', '1120': 'e8a7f304724fc6f3b555ec4f4fb1e6d898a7d78654f499e2ec34d9f2dc903760', '1183': 'e8a7f304724fc6f3b555ec4f4fb1e6d898a7d78654f499e2ec34d9f2dc903760', '1226': '76a3be9f6c9bd644d40686c21a2d6b0b0a3dde451f94aa27785c1cddba09bdb5', '1227': '9e60b030e3fc3416bad089537b6e1b323d5ef6d06e9f66546b63d156fd4ffa45', '1228': '9e8d102118b06dd9dd4dcd19b53b12f240b3467cbc5536024c255fcb998bbb75', '1232': '6139ad1bbc12a6a43b16ab1132f4b786a8d696c901879707f87091bd5b7de8fe', '1253': '76a3be9f6c9bd644d40686c21a2d6b0b0a3dde451f94aa27785c1cddba09bdb5', '1254': '9e60b030e3fc3416bad089537b6e1b323d5ef6d06e9f66546b63d156fd4ffa45', '1259': 'f0ff3a33e71cba52b0192fbc1a682602a12b8f80aff7c82e8da4c125d4effc4c', '1291': 'a730cea9827bc512da8df709301c5767a7817dcfc9b71ef55226ee7aa305806e', '1302': '9f9d188d1d315f54e217d378a057ebd992dcb664c83e153f6bb21980cd61cc9a', '1347': 'bc1c305a86e67f54f4b13bd2d7da163a1a16587394e34be9a809a4f2e4ed0491', '1462': 'e8a7f304724fc6f3b555ec4f4fb1e6d898a7d78654f499e2ec34d9f2dc903760', '1507': 'fdd5306bebb0ff517de97a6f29da43c837ffe1ac40f8f1f9d7326e98a9d357f7', '1516': '93bc88d036a697d382446544594c15b2742ab90c25e44517a263621004aeb447', '1549': '8995f1cb1a69995bd7f684b9a46c2b169c1f2c32c2598dbeb1f10a8adde56a6b', '1568': '57746ec02ffef0f44aa1e6ad248932e7e179d2f190a2b091694fe8e4d8c4b4d6', '1569': '9e60b030e3fc3416bad089537b6e1b323d5ef6d06e9f66546b63d156fd4ffa45', '1574': '3dcb43a7d32fa6f1c227100d15b315db8519fcef7878df9dee34f79c39d84d1d', '1608': 'e463ca37a4791385d46a0476fd36206e09ab1d5b5539fe8f5897a838d58b3fda', '1681': 'cbe66c489d14efa1afe724e83d6626552794ade12c7fb1e076a0c8bf1ae232cf', '1690': '8bfc6370594896baca383df1510782b21c65050ff626bbe52ac11dd2890575d4', '1784': '064f61c7ac128274379b764a540a0b4ff34554324d0b27e47c0d68ca738d25e3', '1785': '142758aff462348a4a4b9a8d6afb4e61c4ac4b6a71ccdbce6855192ce0845895', '1786': '93bc88d036a697d382446544594c15b2742ab90c25e44517a263621004aeb447', '1790': 'f9678ba9e852139905d532f36764bf01da9442302322dc859fdec0413762e855', '1806': 'd619f4eb8d127911138c946f5b7a2c7849cc652acc3caf412ba92daaa0ced3f5', '1843': '7ef73c2a8cedac8e2a1da98324d03e87fb0ad835491cc6bf6050138ea778294a', '1884': '142758aff462348a4a4b9a8d6afb4e61c4ac4b6a71ccdbce6855192ce0845895', '1885': '93bc88d036a697d382446544594c15b2742ab90c25e44517a263621004aeb447', '1889': '7ee215fd727ced2f056600f84cfb3e901f64b39fe309aae820281838f68e33a7', '1892': '4a026ced5812ddd74e65d8ea766e096f043435075c6a53705cc8c898f3983e5f', '1893': '142758aff462348a4a4b9a8d6afb4e61c4ac4b6a71ccdbce6855192ce0845895', '1898': '3dcb43a7d32fa6f1c227100d15b315db8519fcef7878df9dee34f79c39d84d1d', '1918': '1a019a64a45e235c260637243053c31a84369338428d8bd4c58822942a7490e6', '1939': '70dba65866edff0ab960e98994ae7b784c94fe5670b68b5b8568a06bb10d08cf', '1965': '142758aff462348a4a4b9a8d6afb4e61c4ac4b6a71ccdbce6855192ce0845895', '1970': '3dcb43a7d32fa6f1c227100d15b315db8519fcef7878df9dee34f79c39d84d1d', '1984': 'e8a7f304724fc6f3b555ec4f4fb1e6d898a7d78654f499e2ec34d9f2dc903760', '1987': '404c24d0241b492855b677bcf7562180dc256ca5d37aafb7c2ba544c63008dda', '1992': '83f53b49c08848bcc25b31738ac9d1a48cfb9aa0bf44bef277d40226f0a233c5', '1997': '43a2c52db0ed007ab2da12f6471c3cef0d701c9b90d33cd727121b458c420d52', '2095': 'f84931a326e9174c835b0fea17f19a89f4e1dc5fc7b1c9e1058ccfd73ddaa8d1', '2098': 'f31ccacd56cffd006b148b9d40ed35bd8c5290a1c056fed2c9aca93e0729ad78', '2117': '632ca85f527ee3540aa1f1e89a19f6df145215636aeac3c2b7e17321794e7834', '2118': '83f53b49c08848bcc25b31738ac9d1a48cfb9aa0bf44bef277d40226f0a233c5', '2123': '3dcb43a7d32fa6f1c227100d15b315db8519fcef7878df9dee34f79c39d84d1d', '2146': '1297c63eae1002008ad50849f13685208906448bd4157b56facb3e9b84130ec3', '2182': '1297c63eae1002008ad50849f13685208906448bd4157b56facb3e9b84130ec3', '2283': '071fac6713630f7ee51c0ce526843e16c6d50684b0075f8ba8d4049ed2fe3c6e', '2287': '81ea6b49dbb9f7a97917edab53a1b3d98d0c45a6d797eece37011155a10e6d25', '2299': '93bc88d036a697d382446544594c15b2742ab90c25e44517a263621004aeb447', '2346': 'b2761e7124f5210d05b8c6697377e9cd1078fa120868d6ebd222fcb713f71847', '2352': '9e60b030e3fc3416bad089537b6e1b323d5ef6d06e9f66546b63d156fd4ffa45', '2357': 'f0ff3a33e71cba52b0192fbc1a682602a12b8f80aff7c82e8da4c125d4effc4c', '2371': '93bc88d036a697d382446544594c15b2742ab90c25e44517a263621004aeb447', '2377': '632ca85f527ee3540aa1f1e89a19f6df145215636aeac3c2b7e17321794e7834', '2409': '6841da7e16dca6506be6321de7ac3ccc1e68a39bf44037c234a648a4e8973a79', '2434': '9e8d102118b06dd9dd4dcd19b53b12f240b3467cbc5536024c255fcb998bbb75', '2478': '142758aff462348a4a4b9a8d6afb4e61c4ac4b6a71ccdbce6855192ce0845895', '2483': '3dcb43a7d32fa6f1c227100d15b315db8519fcef7878df9dee34f79c39d84d1d', '2495': '4ef9d485257a7e052b8731e0e7cecd2d1acfd13f5d4e9f3ed9131f5cbd2dd15f', '2496': '9e60b030e3fc3416bad089537b6e1b323d5ef6d06e9f66546b63d156fd4ffa45', '2501': '3dcb43a7d32fa6f1c227100d15b315db8519fcef7878df9dee34f79c39d84d1d', '2522': '4ef9d485257a7e052b8731e0e7cecd2d1acfd13f5d4e9f3ed9131f5cbd2dd15f', '2523': '142758aff462348a4a4b9a8d6afb4e61c4ac4b6a71ccdbce6855192ce0845895', '2528': 'f9678ba9e852139905d532f36764bf01da9442302322dc859fdec0413762e855', '2533': '1297c63eae1002008ad50849f13685208906448bd4157b56facb3e9b84130ec3', '2560': 'e8a7f304724fc6f3b555ec4f4fb1e6d898a7d78654f499e2ec34d9f2dc903760', '2587': 'fdd5306bebb0ff517de97a6f29da43c837ffe1ac40f8f1f9d7326e98a9d357f7', '2590': 'f4ef6b953a9f6bf40d510aaff85eec93660bcdf8dd6088f63a3aa35dc84ee9ca', '2604': '9e60b030e3fc3416bad089537b6e1b323d5ef6d06e9f66546b63d156fd4ffa45', '2609': '3dcb43a7d32fa6f1c227100d15b315db8519fcef7878df9dee34f79c39d84d1d', '2616': '752376fc6dd14ae94826c4bda2dd674ec58ed5dae7af14486cd1bfdcc8e734ec', '2631': '9e60b030e3fc3416bad089537b6e1b323d5ef6d06e9f66546b63d156fd4ffa45', '2632': '70dba65866edff0ab960e98994ae7b784c94fe5670b68b5b8568a06bb10d08cf', '2636': 'f9678ba9e852139905d532f36764bf01da9442302322dc859fdec0413762e855', '2638': 'c25b973ecbc999655609b6b1817c96183cade12ab04dd0cfa90556cfcce60ae3', '2661': '4de811054dc4baa772ec7887d4805ec6f42e3b4b3c297df159e54b88db795d26', '2667': '83f53b49c08848bcc25b31738ac9d1a48cfb9aa0bf44bef277d40226f0a233c5', '2670': 'f7b4eb183a86336ddf789c9eccfb9fd71c1631a4c377b38fa435c53f0166cb93', '2672': '3dcb43a7d32fa6f1c227100d15b315db8519fcef7878df9dee34f79c39d84d1d', '2693': '86adf05ad387ed0f2922d6e40a819aeba74ef8d51fc16c62eaccdedb007ee529', '2694': '9e60b030e3fc3416bad089537b6e1b323d5ef6d06e9f66546b63d156fd4ffa45', '2699': 'cdb9d08f8d29b32ab5384800aee61346b317c5343582918e57206c4abc79955a', '2719': '8995f1cb1a69995bd7f684b9a46c2b169c1f2c32c2598dbeb1f10a8adde56a6b', '2766': '9e60b030e3fc3416bad089537b6e1b323d5ef6d06e9f66546b63d156fd4ffa45', '2771': '3dcb43a7d32fa6f1c227100d15b315db8519fcef7878df9dee34f79c39d84d1d', '2787': 'bb9119c99068bc7b14654af5e77821b68547eedc1dda0a5efcfba0fc39eb83bb', '2794': '1297c63eae1002008ad50849f13685208906448bd4157b56facb3e9b84130ec3', '2810': '6f9d5c5a89c44bffad714d20b0a432b99388cdb66836c54d118172630ab07223', '2811': '83f53b49c08848bcc25b31738ac9d1a48cfb9aa0bf44bef277d40226f0a233c5', '2816': 'f9678ba9e852139905d532f36764bf01da9442302322dc859fdec0413762e855', '2827': '57aed967860902e54ed6ac43cf9d63280f500ac61a48a42c09adbac628bf4e18', '2859': 'b2761e7124f5210d05b8c6697377e9cd1078fa120868d6ebd222fcb713f71847', '2865': '142758aff462348a4a4b9a8d6afb4e61c4ac4b6a71ccdbce6855192ce0845895', '2866': '1297c63eae1002008ad50849f13685208906448bd4157b56facb3e9b84130ec3', '2870': 'cdb9d08f8d29b32ab5384800aee61346b317c5343582918e57206c4abc79955a', '2873': '57746ec02ffef0f44aa1e6ad248932e7e179d2f190a2b091694fe8e4d8c4b4d6', '2874': '83f53b49c08848bcc25b31738ac9d1a48cfb9aa0bf44bef277d40226f0a233c5', '2879': 'f9678ba9e852139905d532f36764bf01da9442302322dc859fdec0413762e855', '2893': '93bc88d036a697d382446544594c15b2742ab90c25e44517a263621004aeb447', '2938': '70dba65866edff0ab960e98994ae7b784c94fe5670b68b5b8568a06bb10d08cf', '2946': '83f53b49c08848bcc25b31738ac9d1a48cfb9aa0bf44bef277d40226f0a233c5', '2951': 'f0ff3a33e71cba52b0192fbc1a682602a12b8f80aff7c82e8da4c125d4effc4c', '2954': 'cd70225a5ad47a8ecbf6214f41556d835194dadd34e70d15801528a865d9af14', '2955': '83f53b49c08848bcc25b31738ac9d1a48cfb9aa0bf44bef277d40226f0a233c5', '2959': '93b105767530d1f122dfde4a95f3480b891761618426b54deba47c94318b6465', '2960': 'f9678ba9e852139905d532f36764bf01da9442302322dc859fdec0413762e855', '2977': 'faf3d5008a3563f5706f29a06cd3c7981b169951e2448b4e5f41e679b1e1256b', '2994': '3e5d4da88fd3764e3a1c4d4a27af5e701f657db939f0ba63619a561586d5a152', '3028': 'fdd5306bebb0ff517de97a6f29da43c837ffe1ac40f8f1f9d7326e98a9d357f7', '3037': '70dba65866edff0ab960e98994ae7b784c94fe5670b68b5b8568a06bb10d08cf', '3094': 'cfff93e2b23f841b68ea37117f86bbd275a6aa839063927645ccb05ba82f2012', '3116': '9647e99b6ce78ff73f6ebb5f25dbd8384e84361bf9f8e677b2325a9de36b3d99', '3117': '142758aff462348a4a4b9a8d6afb4e61c4ac4b6a71ccdbce6855192ce0845895', '3120': '93b105767530d1f122dfde4a95f3480b891761618426b54deba47c94318b6465', '3122': 'f0ff3a33e71cba52b0192fbc1a682602a12b8f80aff7c82e8da4c125d4effc4c', '3235': '1297c63eae1002008ad50849f13685208906448bd4157b56facb3e9b84130ec3', '3246': 'd00eea2d79249b65db22aed36e646cf30bc6cbb46e63c7ade8efdca131626944', '3250': 'd2e133de6ebbb8b558eb90d2e7b407c121808c9220571fd1f4c37d59aa9fb2ef', '3332': 'b43d75e58feb5bec6af55bf31e3f8df270870b0ff26cc19aeb84de4bd2825d5d', '3333': '83f53b49c08848bcc25b31738ac9d1a48cfb9aa0bf44bef277d40226f0a233c5', '3338': 'cdb9d08f8d29b32ab5384800aee61346b317c5343582918e57206c4abc79955a', '3341': '211af9931088b5b0014efee3ac8cfe2efce059d5935d11d3aa6551b55e1e6046', '3342': '9e60b030e3fc3416bad089537b6e1b323d5ef6d06e9f66546b63d156fd4ffa45', '3347': 'f9678ba9e852139905d532f36764bf01da9442302322dc859fdec0413762e855', '3417': '666166e6db8ed75a77d1916f3f500cb21fbf04b40de5c82d590b3fe00f56e392', '3441': '9e60b030e3fc3416bad089537b6e1b323d5ef6d06e9f66546b63d156fd4ffa45', '3446': 'cdb9d08f8d29b32ab5384800aee61346b317c5343582918e57206c4abc79955a', '3505': 'e8a7f304724fc6f3b555ec4f4fb1e6d898a7d78654f499e2ec34d9f2dc903760', '3532': '29b15925ddf98a7d2d3834102e2ad99fc4187fe0dd6fea1319b561a3d34403f3', '3538': 'c25b973ecbc999655609b6b1817c96183cade12ab04dd0cfa90556cfcce60ae3', '3574': '81ea6b49dbb9f7a97917edab53a1b3d98d0c45a6d797eece37011155a10e6d25', '3624': '1790c8bba94347044990fd6a16ca7c8e1c69ea7d355c6c12ff05fd4fbfcaea26', '3700': '1a019a64a45e235c260637243053c31a84369338428d8bd4c58822942a7490e6', '3701': '86adf05ad387ed0f2922d6e40a819aeba74ef8d51fc16c62eaccdedb007ee529', '3702': '142758aff462348a4a4b9a8d6afb4e61c4ac4b6a71ccdbce6855192ce0845895', '3707': '3dcb43a7d32fa6f1c227100d15b315db8519fcef7878df9dee34f79c39d84d1d', '3739': '93bc88d036a697d382446544594c15b2742ab90c25e44517a263621004aeb447', '3768': 'b92698d75a9ab158a830f728599f133338f307f2927c87fae0fb1db32214bf60', '3781': '86adf05ad387ed0f2922d6e40a819aeba74ef8d51fc16c62eaccdedb007ee529', '3822': '1890331f8082b9f7bf8d1094aafa7ad95ae9d4dda5896c356124e1e0c20244a4', '3838': 'a730cea9827bc512da8df709301c5767a7817dcfc9b71ef55226ee7aa305806e', '3858': 'e6f3bf22ea5d1987b65ebf8e4c18e27ea9ef1833241b76a9e20bc94a6d8487b3', '3867': 'a28f33098a0ac2ddb5778b78d16adc515d0964612b2e76a4077bd19f99bcf0eb', '3927': '142758aff462348a4a4b9a8d6afb4e61c4ac4b6a71ccdbce6855192ce0845895', '3932': 'cdb9d08f8d29b32ab5384800aee61346b317c5343582918e57206c4abc79955a', '3953': '81ea6b49dbb9f7a97917edab53a1b3d98d0c45a6d797eece37011155a10e6d25', '3954': '9e60b030e3fc3416bad089537b6e1b323d5ef6d06e9f66546b63d156fd4ffa45', '3955': '70dba65866edff0ab960e98994ae7b784c94fe5670b68b5b8568a06bb10d08cf', '3959': 'cdb9d08f8d29b32ab5384800aee61346b317c5343582918e57206c4abc79955a', '3973': '4a057ea52e7cdfb39aa6bcc2a4115e00457830433f665561a698deba66d84b86', '3997': '166d3d092858114a1f8c3215e6570f05c51f07a2111b5f0103c8ce8a00d7ea11', '4020': '1bb7e36a34fa3352027fd71fe08215a0c8d1e58dcbee3117aa643489b54ca0d1', '4036': '93bc88d036a697d382446544594c15b2742ab90c25e44517a263621004aeb447', '4062': '9e60b030e3fc3416bad089537b6e1b323d5ef6d06e9f66546b63d156fd4ffa45', '4067': 'f9678ba9e852139905d532f36764bf01da9442302322dc859fdec0413762e855', '4093': '63c791f2f9b17ecca6913a5612bbcd29305ffdfad611ccc445a360ae5fabb782', '4105': '81ea6b49dbb9f7a97917edab53a1b3d98d0c45a6d797eece37011155a10e6d25', '4108': '93bc88d036a697d382446544594c15b2742ab90c25e44517a263621004aeb447', '4204': 'f1fbbdd03a1a041e9a85abc83d788e8617ccdb64817df0b15e26c3ec1bc337b0', '4222': '3c987826c2a9b67a3bbc34e4577949b3d3142ef61bd78b9ce3f535e745101e5c', '4254': 'c4a2c3f1198266cc8f5b33dfb176912ed0d2bd35aa8c140ac3627e969e2c037a', '4270': '9e8d102118b06dd9dd4dcd19b53b12f240b3467cbc5536024c255fcb998bbb75', '4282': '43282774e02635253f070554e3ea703f7ab4eef42e061b0a9b445edde6cb2915', '4351': 'fdd5306bebb0ff517de97a6f29da43c837ffe1ac40f8f1f9d7326e98a9d357f7', '4363': '0c0b3840e87d31d9e6d1d527e7c454aa64231499dd2dfc9330d9838523276609', '4366': 'd637fd9061dfb0384dff2488eb6e4761f1a8a110e0b556a58f0ff863fd83fdb6', '4396': '93bc88d036a697d382446544594c15b2742ab90c25e44517a263621004aeb447', '4440': '9e60b030e3fc3416bad089537b6e1b323d5ef6d06e9f66546b63d156fd4ffa45', '4445': 'f0ff3a33e71cba52b0192fbc1a682602a12b8f80aff7c82e8da4c125d4effc4c', '4470': 'ce917b851dbc5b60a6fcb1c0c7303f3d09570f150dabfe282ac356b85b317444', '4493': '2f50191604f44ee0b14e81222d4c8a8bb0e4e401df9feea99b18a18b9c36ce77', '4494': '142758aff462348a4a4b9a8d6afb4e61c4ac4b6a71ccdbce6855192ce0845895', '4499': 'f0ff3a33e71cba52b0192fbc1a682602a12b8f80aff7c82e8da4c125d4effc4c', '4522': 'e8a7f304724fc6f3b555ec4f4fb1e6d898a7d78654f499e2ec34d9f2dc903760', '4552': 'de23e9f2c88a3b7ff57ee12b267d71d3fa95b10c5e86635a6dfb8aa20e80a22a', '4588': '23a3c9cdaeadc6da53ea4b284389e675a9715ef29e5c6ced670309bb3770eab5', '4619': '6aee511592728d9e017318e9fdc558a810b79dccf1cbad353ebe3e4420b8deaf', '4620': '142758aff462348a4a4b9a8d6afb4e61c4ac4b6a71ccdbce6855192ce0845895', '4625': 'cdb9d08f8d29b32ab5384800aee61346b317c5343582918e57206c4abc79955a', '4672': '8cdd6b986a9ac1bb3c649819f95489893673b064afed0a2197ff7c8e86055da8', '4691': 'f1fbbdd03a1a041e9a85abc83d788e8617ccdb64817df0b15e26c3ec1bc337b0', '4692': '83f53b49c08848bcc25b31738ac9d1a48cfb9aa0bf44bef277d40226f0a233c5', '4697': 'f0ff3a33e71cba52b0192fbc1a682602a12b8f80aff7c82e8da4c125d4effc4c', '4701': '83f53b49c08848bcc25b31738ac9d1a48cfb9aa0bf44bef277d40226f0a233c5', '4706': 'f9678ba9e852139905d532f36764bf01da9442302322dc859fdec0413762e855', '4741': '2dc39d9e385ddf99a7f11a2a55be83971fd2d45ed9e6090a9f7d063d73d8c129', '4745': '6aee511592728d9e017318e9fdc558a810b79dccf1cbad353ebe3e4420b8deaf', '4746': '9e60b030e3fc3416bad089537b6e1b323d5ef6d06e9f66546b63d156fd4ffa45', '4751': 'f9678ba9e852139905d532f36764bf01da9442302322dc859fdec0413762e855', '4758': '7781a4db8a02fedcc9d9ba029346ca2533d0e3d0969359189228485836085f8c', '4776': 'bbfc10cb31748cfcc1b3ee55d8b0c4525b74f956a205fd15f8e5481b4270cc2b', '4789': '064f61c7ac128274379b764a540a0b4ff34554324d0b27e47c0d68ca738d25e3', '4792': '93bc88d036a697d382446544594c15b2742ab90c25e44517a263621004aeb447', '4801': '70dba65866edff0ab960e98994ae7b784c94fe5670b68b5b8568a06bb10d08cf', '4812': '11958a31813a0732f50ddc8e8d3fad636a57e2733510ef0da7fb8331df694bb9', '4819': '1297c63eae1002008ad50849f13685208906448bd4157b56facb3e9b84130ec3', '4828': 'fdd5306bebb0ff517de97a6f29da43c837ffe1ac40f8f1f9d7326e98a9d357f7', '4837': '1297c63eae1002008ad50849f13685208906448bd4157b56facb3e9b84130ec3', '4846': '93bc88d036a697d382446544594c15b2742ab90c25e44517a263621004aeb447', '4881': '9e60b030e3fc3416bad089537b6e1b323d5ef6d06e9f66546b63d156fd4ffa45', '4886': '3dcb43a7d32fa6f1c227100d15b315db8519fcef7878df9dee34f79c39d84d1d', '4890': '142758aff462348a4a4b9a8d6afb4e61c4ac4b6a71ccdbce6855192ce0845895', '4893': '6dfae581f7bf1b2d590081c13bfa63167677da899bf3eee872cf2533e7412fb6', '4895': 'f9678ba9e852139905d532f36764bf01da9442302322dc859fdec0413762e855', '4927': 'e8a7f304724fc6f3b555ec4f4fb1e6d898a7d78654f499e2ec34d9f2dc903760', '4935': '83f53b49c08848bcc25b31738ac9d1a48cfb9aa0bf44bef277d40226f0a233c5', '4940': '3dcb43a7d32fa6f1c227100d15b315db8519fcef7878df9dee34f79c39d84d1d', '4943': '241702f1bf26ed11a2f80ada3248f6d39bae10ba1dd894c91a2da085f0070f9f', '4944': '9e60b030e3fc3416bad089537b6e1b323d5ef6d06e9f66546b63d156fd4ffa45', '4949': 'cdb9d08f8d29b32ab5384800aee61346b317c5343582918e57206c4abc79955a', '4966': '863286368787f979f892948278b3a26838ad08db69cec2421ea1a5a52c1ddf31', '5019': '0fbe1189ca46f67ef7595e3707020411c0c36af412a48d53b3d9f664462fc341', '5020': '7a3417da199369a1bcac95d9493b883f1f71d58026860a260f6c2ce6803a2682', '5059': '0b2289a5b1117dcbdaedaab8311f802d09aa3e931cf3d95b3bc0becad7e29431', '5074': '8abed7d3e4a19baa2abffbd8bf72f41012451281661cb16480a6d524389dde31', '5110': '02684f64624a897c4561360def567a33b98133f9733154fae053c31999b07a2d', '5133': '83f53b49c08848bcc25b31738ac9d1a48cfb9aa0bf44bef277d40226f0a233c5', '5138': '7ee215fd727ced2f056600f84cfb3e901f64b39fe309aae820281838f68e33a7', '5181': '4f2fc0aa3410499710c986c50fab1a38551dee1c4e6e22b394fc42d928f71d2e', '5188': '93bc88d036a697d382446544594c15b2742ab90c25e44517a263621004aeb447', '5199': 'f25045d35cc10222120eda6dfe74aba93fb6a2cc5208b3f228467383fa25c47b', '5205': '142758aff462348a4a4b9a8d6afb4e61c4ac4b6a71ccdbce6855192ce0845895', '5206': '93bc88d036a697d382446544594c15b2742ab90c25e44517a263621004aeb447', '5210': 'f9678ba9e852139905d532f36764bf01da9442302322dc859fdec0413762e855', '5258': '03d4bc9c784afc70356e45c571ed555b84aef9c667182c76def56782d68aeaf9', '5259': '142758aff462348a4a4b9a8d6afb4e61c4ac4b6a71ccdbce6855192ce0845895', '5264': '3dcb43a7d32fa6f1c227100d15b315db8519fcef7878df9dee34f79c39d84d1d', '5296': '70dba65866edff0ab960e98994ae7b784c94fe5670b68b5b8568a06bb10d08cf', '5304': '9e60b030e3fc3416bad089537b6e1b323d5ef6d06e9f66546b63d156fd4ffa45', '5305': '93bc88d036a697d382446544594c15b2742ab90c25e44517a263621004aeb447', '5309': 'f9678ba9e852139905d532f36764bf01da9442302322dc859fdec0413762e855', '5313': '83f53b49c08848bcc25b31738ac9d1a48cfb9aa0bf44bef277d40226f0a233c5', '5318': 'f0ff3a33e71cba52b0192fbc1a682602a12b8f80aff7c82e8da4c125d4effc4c', '5411': 'c5e526abbbc2e8407146d4620d91fa88f557ecac961745af9438f940afc646ee', '5412': '9e60b030e3fc3416bad089537b6e1b323d5ef6d06e9f66546b63d156fd4ffa45', '5416': '9bcf1b4686c8cdd815b000868b6fec5abc9c1236721e7761b9442f215bd4e7f7', '5417': '3dcb43a7d32fa6f1c227100d15b315db8519fcef7878df9dee34f79c39d84d1d', '5431': '93bc88d036a697d382446544594c15b2742ab90c25e44517a263621004aeb447', '5439': '9e60b030e3fc3416bad089537b6e1b323d5ef6d06e9f66546b63d156fd4ffa45', '5444': 'f0ff3a33e71cba52b0192fbc1a682602a12b8f80aff7c82e8da4c125d4effc4c', '5451': 'a6bb245a604b1d67736ef843907e6f5dd7c10f05fbda7a7075979949d20f6e87', '5479': 'cdcd1423892aa443e0db3b7b509b19cf6a3eb4f9dd942ceae8a62747c520675a', '5556': '142758aff462348a4a4b9a8d6afb4e61c4ac4b6a71ccdbce6855192ce0845895', '5561': 'cdb9d08f8d29b32ab5384800aee61346b317c5343582918e57206c4abc79955a', '5591': '8ec429e69a672bc17eb6765695fec5af188794f1c4402e937fe3d871684e2000', '5592': '142758aff462348a4a4b9a8d6afb4e61c4ac4b6a71ccdbce6855192ce0845895', '5597': 'f9678ba9e852139905d532f36764bf01da9442302322dc859fdec0413762e855', '5636': 'd2e133de6ebbb8b558eb90d2e7b407c121808c9220571fd1f4c37d59aa9fb2ef', '5637': '83f53b49c08848bcc25b31738ac9d1a48cfb9aa0bf44bef277d40226f0a233c5', '5642': '3dcb43a7d32fa6f1c227100d15b315db8519fcef7878df9dee34f79c39d84d1d', '5647': '1297c63eae1002008ad50849f13685208906448bd4157b56facb3e9b84130ec3', '5658': 'd88dc5b2312b12ff4f38e44bd6a89c62ac7572d311d2c4ca4d7e45ce55fc1d98', '5676': '7e62f01f28264343b20d365ba591af71226bc690bb81b5cadd612207e5d0cb78', '5698': '1a019a64a45e235c260637243053c31a84369338428d8bd4c58822942a7490e6', '5726': 'cd70225a5ad47a8ecbf6214f41556d835194dadd34e70d15801528a865d9af14', '5727': '142758aff462348a4a4b9a8d6afb4e61c4ac4b6a71ccdbce6855192ce0845895', '5732': 'f9678ba9e852139905d532f36764bf01da9442302322dc859fdec0413762e855', '5764': '70dba65866edff0ab960e98994ae7b784c94fe5670b68b5b8568a06bb10d08cf', '5836': '93bc88d036a697d382446544594c15b2742ab90c25e44517a263621004aeb447', '5871': '83f53b49c08848bcc25b31738ac9d1a48cfb9aa0bf44bef277d40226f0a233c5', '5876': 'cdb9d08f8d29b32ab5384800aee61346b317c5343582918e57206c4abc79955a', '5942': '4a026ced5812ddd74e65d8ea766e096f043435075c6a53705cc8c898f3983e5f', '5943': '9e60b030e3fc3416bad089537b6e1b323d5ef6d06e9f66546b63d156fd4ffa45', '5944': '93bc88d036a697d382446544594c15b2742ab90c25e44517a263621004aeb447', '5948': '3dcb43a7d32fa6f1c227100d15b315db8519fcef7878df9dee34f79c39d84d1d', '5961': '9e60b030e3fc3416bad089537b6e1b323d5ef6d06e9f66546b63d156fd4ffa45', '5966': 'cdb9d08f8d29b32ab5384800aee61346b317c5343582918e57206c4abc79955a', '5971': '29b15925ddf98a7d2d3834102e2ad99fc4187fe0dd6fea1319b561a3d34403f3', '6034': '70dba65866edff0ab960e98994ae7b784c94fe5670b68b5b8568a06bb10d08cf', '6036': '4bd8b5e0428c3811535f9aadcb018111f32e3ea85981249216b3aed4430eb0e7', '6104': '3c987826c2a9b67a3bbc34e4577949b3d3142ef61bd78b9ce3f535e745101e5c', '6105': '9e60b030e3fc3416bad089537b6e1b323d5ef6d06e9f66546b63d156fd4ffa45', '6110': '3dcb43a7d32fa6f1c227100d15b315db8519fcef7878df9dee34f79c39d84d1d', '6114': '142758aff462348a4a4b9a8d6afb4e61c4ac4b6a71ccdbce6855192ce0845895', '6119': '3dcb43a7d32fa6f1c227100d15b315db8519fcef7878df9dee34f79c39d84d1d', '6133': '70dba65866edff0ab960e98994ae7b784c94fe5670b68b5b8568a06bb10d08cf', '6166': '96782cf86ff795f5afd96320c5dd56b303607d13c34c2fe8877711fd5220e268', '6185': '28a3fbf18484f0915a2b6c50f699de53ec840492940ea92f07558e5d64acb946', '6186': '83f53b49c08848bcc25b31738ac9d1a48cfb9aa0bf44bef277d40226f0a233c5', '6191': 'cdb9d08f8d29b32ab5384800aee61346b317c5343582918e57206c4abc79955a', '6229': '19022d96f8804e985a361e8e9adf98c5714e3025b5b063760dc60e8e68633c4f', '6231': '83f53b49c08848bcc25b31738ac9d1a48cfb9aa0bf44bef277d40226f0a233c5', '6236': 'cdb9d08f8d29b32ab5384800aee61346b317c5343582918e57206c4abc79955a', '6292': 'b3a9e32c904d728719eab27656cae4a3c769c2b5d2424f45c15cedc3a4fe6fb2', '6421': '1297c63eae1002008ad50849f13685208906448bd4157b56facb3e9b84130ec3', '6429': '142758aff462348a4a4b9a8d6afb4e61c4ac4b6a71ccdbce6855192ce0845895', '6434': 'f0ff3a33e71cba52b0192fbc1a682602a12b8f80aff7c82e8da4c125d4effc4c', '6562': '1bd9bbe48f1ee49b11dd7d8ab67d393c8c45ba7ea0630f07a1d4150102fd3978', '6585': '7d58f0e89263f350665b9ae812d5c0972d64f44f13fe8ecfc78934fb69c0fb61', '6635': 'cd70225a5ad47a8ecbf6214f41556d835194dadd34e70d15801528a865d9af14', '6636': '9e60b030e3fc3416bad089537b6e1b323d5ef6d06e9f66546b63d156fd4ffa45', '6641': '3dcb43a7d32fa6f1c227100d15b315db8519fcef7878df9dee34f79c39d84d1d', '6643': '211af9931088b5b0014efee3ac8cfe2efce059d5935d11d3aa6551b55e1e6046', '6649': '7deb196e7bbfc1879f3d894c3fd780c4bc139d8bb6ae7ef24bc24662d9c3f5d6', '6652': '19022d96f8804e985a361e8e9adf98c5714e3025b5b063760dc60e8e68633c4f', '6662': '19022d96f8804e985a361e8e9adf98c5714e3025b5b063760dc60e8e68633c4f', '6663': '83f53b49c08848bcc25b31738ac9d1a48cfb9aa0bf44bef277d40226f0a233c5', '6668': 'f0ff3a33e71cba52b0192fbc1a682602a12b8f80aff7c82e8da4c125d4effc4c', '6672': '142758aff462348a4a4b9a8d6afb4e61c4ac4b6a71ccdbce6855192ce0845895', '6677': 'f0ff3a33e71cba52b0192fbc1a682602a12b8f80aff7c82e8da4c125d4effc4c', '6706': 'b43d75e58feb5bec6af55bf31e3f8df270870b0ff26cc19aeb84de4bd2825d5d', '6780': '9e60b030e3fc3416bad089537b6e1b323d5ef6d06e9f66546b63d156fd4ffa45', '6781': '93bc88d036a697d382446544594c15b2742ab90c25e44517a263621004aeb447', '6785': '3dcb43a7d32fa6f1c227100d15b315db8519fcef7878df9dee34f79c39d84d1d', '6817': '70dba65866edff0ab960e98994ae7b784c94fe5670b68b5b8568a06bb10d08cf', '6844': '70dba65866edff0ab960e98994ae7b784c94fe5670b68b5b8568a06bb10d08cf', '6851': 'c25b973ecbc999655609b6b1817c96183cade12ab04dd0cfa90556cfcce60ae3', '6852': '83f53b49c08848bcc25b31738ac9d1a48cfb9aa0bf44bef277d40226f0a233c5', '6857': '17a0916a59b078ec4dafecb6804e0a74a7c224cde4ccdbe91a47f0b50d863431', '6864': '9c0cae88f95340ed79892f342499cfc0f73d6bcc0817cea244dee2b1cac54eb2', '6879': '142758aff462348a4a4b9a8d6afb4e61c4ac4b6a71ccdbce6855192ce0845895', '6884': 'f0ff3a33e71cba52b0192fbc1a682602a12b8f80aff7c82e8da4c125d4effc4c', '6910': '7a3417da199369a1bcac95d9493b883f1f71d58026860a260f6c2ce6803a2682', '6961': '93bc88d036a697d382446544594c15b2742ab90c25e44517a263621004aeb447', '6970': '9e8d102118b06dd9dd4dcd19b53b12f240b3467cbc5536024c255fcb998bbb75', '6972': 'a703e515c47e52dcd144c15989b27d336ba9825e161e638537d80991851a1ea4', '7006': '1297c63eae1002008ad50849f13685208906448bd4157b56facb3e9b84130ec3', '7086': '142758aff462348a4a4b9a8d6afb4e61c4ac4b6a71ccdbce6855192ce0845895', '7089': '93b105767530d1f122dfde4a95f3480b891761618426b54deba47c94318b6465', '7091': 'cdb9d08f8d29b32ab5384800aee61346b317c5343582918e57206c4abc79955a', '7096': '1297c63eae1002008ad50849f13685208906448bd4157b56facb3e9b84130ec3', '7117': '26f60ae707d236c8bf649de249110c60fcaf4aeb5f8af2535bf0537e040dd300', '7141': '1297c63eae1002008ad50849f13685208906448bd4157b56facb3e9b84130ec3', '7159': '93bc88d036a697d382446544594c15b2742ab90c25e44517a263621004aeb447', '7188': '1439f9d870bd29e3d652cb7bed09196feb791d8021bd80de2ac746f9205fd9d3', '7195': '93bc88d036a697d382446544594c15b2742ab90c25e44517a263621004aeb447', '7207': '73005245ed8de5d35b458badc8392937a2caae60ef9219bf27479d439ba952c3', '7301': '8995f1cb1a69995bd7f684b9a46c2b169c1f2c32c2598dbeb1f10a8adde56a6b', '7302': '83f53b49c08848bcc25b31738ac9d1a48cfb9aa0bf44bef277d40226f0a233c5', '7307': '43a2c52db0ed007ab2da12f6471c3cef0d701c9b90d33cd727121b458c420d52', '7309': 'b43d75e58feb5bec6af55bf31e3f8df270870b0ff26cc19aeb84de4bd2825d5d', '7366': '93bc88d036a697d382446544594c15b2742ab90c25e44517a263621004aeb447', '7378': '8234523848c5e11862694dab066f3a038d458934248e0fa29921517413292af6', '7383': '9e60b030e3fc3416bad089537b6e1b323d5ef6d06e9f66546b63d156fd4ffa45', '7388': 'f9678ba9e852139905d532f36764bf01da9442302322dc859fdec0413762e855', '7437': '9e60b030e3fc3416bad089537b6e1b323d5ef6d06e9f66546b63d156fd4ffa45', '7442': 'cdb9d08f8d29b32ab5384800aee61346b317c5343582918e57206c4abc79955a', '7447': '93bc88d036a697d382446544594c15b2742ab90c25e44517a263621004aeb447', '7495': '8ce998747f3e3183fe31741e346272972701d02b7bf48d88e38b49690fe102b8', '7519': '93bc88d036a697d382446544594c15b2742ab90c25e44517a263621004aeb447', '7535': 'f1fbbdd03a1a041e9a85abc83d788e8617ccdb64817df0b15e26c3ec1bc337b0', '7536': '83f53b49c08848bcc25b31738ac9d1a48cfb9aa0bf44bef277d40226f0a233c5', '7539': 'e931421159293a769e816ba38c954d4a01a02dc47780249379c952484330d8d6', '7541': '17a0916a59b078ec4dafecb6804e0a74a7c224cde4ccdbe91a47f0b50d863431', '7612': 'f8abe02e66556084f1207fac8b00874ad55267dfd42f3cffad219867b6ad54ea', '7648': '64afc722ecc190656736935de793930e7f2fb3210a7499841a2d1e56616c0d70', '7660': '57aed967860902e54ed6ac43cf9d63280f500ac61a48a42c09adbac628bf4e18', '7693': 'b501968b0e620f182bb40dea9a1f094d70ce5fae9b5e2ef30df835d42293aed1', '7696': '2f50191604f44ee0b14e81222d4c8a8bb0e4e401df9feea99b18a18b9c36ce77', '7699': '70dba65866edff0ab960e98994ae7b784c94fe5670b68b5b8568a06bb10d08cf', '7710': 'e0f04e486c062ac9ba95ca2e7dbb411b54b04aa19cf8ddd01300fffc1fda1f13', '7714': '867e140889f668f32ed79e7c1f455246497c1e4f4d466ccd2e55f49974663701', '7733': '3ce903951b28bb3f1e719f94abfc9e75385c1eef498cc0aec6bec7bb9e1df826', '7734': '83f53b49c08848bcc25b31738ac9d1a48cfb9aa0bf44bef277d40226f0a233c5', '7739': 'f9678ba9e852139905d532f36764bf01da9442302322dc859fdec0413762e855', '7750': '3c987826c2a9b67a3bbc34e4577949b3d3142ef61bd78b9ce3f535e745101e5c', '7797': '142758aff462348a4a4b9a8d6afb4e61c4ac4b6a71ccdbce6855192ce0845895', '7802': 'f9678ba9e852139905d532f36764bf01da9442302322dc859fdec0413762e855', '7807': 'd447f54fe43f904127fbaf528413516c2c1a4ee586f897188d6cefbc61269308', '7825': 'e8a7f304724fc6f3b555ec4f4fb1e6d898a7d78654f499e2ec34d9f2dc903760', '7831': 'c764115323a61e8e547b900c5aff3de1fb06a7c96cf5a1e2688101fd4258b3f1', '7890': 'e47555b70a806d90889ac2a5290117289d3b64e2e84462865ea3ea45c43c507c', '7899': '8ad8c26cbf4041f5c9724d4b29a0a6f43fb4bff2c76237d88c7aaec568edbf02', '7914': '142758aff462348a4a4b9a8d6afb4e61c4ac4b6a71ccdbce6855192ce0845895', '7919': 'f9678ba9e852139905d532f36764bf01da9442302322dc859fdec0413762e855', '7949': '86adf05ad387ed0f2922d6e40a819aeba74ef8d51fc16c62eaccdedb007ee529', '7950': '83f53b49c08848bcc25b31738ac9d1a48cfb9aa0bf44bef277d40226f0a233c5', '7953': 'cb3b83e84311b5385a81d8dc55f3485ca7f855759aac450e6475d1fc1d07035b', '7955': '2d8dd5c46b54b56ec7e6e441bad0a63f3a7124df42e534237daf8bc1bf305bc5', '8005': '70dba65866edff0ab960e98994ae7b784c94fe5670b68b5b8568a06bb10d08cf', '8023': '93bc88d036a697d382446544594c15b2742ab90c25e44517a263621004aeb447', '8062': '705bac025c1d7eb03d67db2bdcc3bcfb298285859d22af12a8ab09f3d872a456', '8093': '19022d96f8804e985a361e8e9adf98c5714e3025b5b063760dc60e8e68633c4f', '8094': '9e60b030e3fc3416bad089537b6e1b323d5ef6d06e9f66546b63d156fd4ffa45', '8099': 'f9678ba9e852139905d532f36764bf01da9442302322dc859fdec0413762e855', '8111': '3ce903951b28bb3f1e719f94abfc9e75385c1eef498cc0aec6bec7bb9e1df826', '8112': '142758aff462348a4a4b9a8d6afb4e61c4ac4b6a71ccdbce6855192ce0845895', '8117': 'f9678ba9e852139905d532f36764bf01da9442302322dc859fdec0413762e855', '8120': '19022d96f8804e985a361e8e9adf98c5714e3025b5b063760dc60e8e68633c4f', '8121': '142758aff462348a4a4b9a8d6afb4e61c4ac4b6a71ccdbce6855192ce0845895', '8126': '17a0916a59b078ec4dafecb6804e0a74a7c224cde4ccdbe91a47f0b50d863431', '8131': '93bc88d036a697d382446544594c15b2742ab90c25e44517a263621004aeb447', '8167': '1297c63eae1002008ad50849f13685208906448bd4157b56facb3e9b84130ec3', '8215': '52af20976ae984b813bc05d6dfe6e1f607ee02d75bbc25ce9995b5fe1799ca28', '8290': '1a019a64a45e235c260637243053c31a84369338428d8bd4c58822942a7490e6', '8293': '70dba65866edff0ab960e98994ae7b784c94fe5670b68b5b8568a06bb10d08cf', '8317': '12cb8533bb086cbc6b9e99163dad6c8ccc8361564c8ec2005262580fb48a683b', '8318': '516a0dd7cd1639f90fa31a6bee23147cae111cfb3c89c34d3d0e701bf8be8647', '8319': '83f53b49c08848bcc25b31738ac9d1a48cfb9aa0bf44bef277d40226f0a233c5', '8324': 'f9678ba9e852139905d532f36764bf01da9442302322dc859fdec0413762e855', '8365': '1297c63eae1002008ad50849f13685208906448bd4157b56facb3e9b84130ec3', '8380': 'c764115323a61e8e547b900c5aff3de1fb06a7c96cf5a1e2688101fd4258b3f1', '8437': 'fdd5306bebb0ff517de97a6f29da43c837ffe1ac40f8f1f9d7326e98a9d357f7', '8444': 'ca8bd20cfe617d890ec1c53c4abfea28ad060e5a35485153e44ccdd4b3a05d3b', '8445': '142758aff462348a4a4b9a8d6afb4e61c4ac4b6a71ccdbce6855192ce0845895', '8450': 'f9678ba9e852139905d532f36764bf01da9442302322dc859fdec0413762e855', '8455': '1297c63eae1002008ad50849f13685208906448bd4157b56facb3e9b84130ec3', '8509': '70dba65866edff0ab960e98994ae7b784c94fe5670b68b5b8568a06bb10d08cf', '8525': 'ca8bd20cfe617d890ec1c53c4abfea28ad060e5a35485153e44ccdd4b3a05d3b', '8526': '83f53b49c08848bcc25b31738ac9d1a48cfb9aa0bf44bef277d40226f0a233c5', '8531': '6139ad1bbc12a6a43b16ab1132f4b786a8d696c901879707f87091bd5b7de8fe', '8563': '93bc88d036a697d382446544594c15b2742ab90c25e44517a263621004aeb447', '8593': 'e47555b70a806d90889ac2a5290117289d3b64e2e84462865ea3ea45c43c507c', '8599': '70dba65866edff0ab960e98994ae7b784c94fe5670b68b5b8568a06bb10d08cf', '8626': '1297c63eae1002008ad50849f13685208906448bd4157b56facb3e9b84130ec3', '8628': 'a6d735879af5cd288e9e5ea90dd1c0b23beef412a37d139daf39d3670dea292b', '8664': '47f74d6611df2ff3aacd078eb125c0ab02470499da6ff4139c5c2158a7021a25', '8704': '1a019a64a45e235c260637243053c31a84369338428d8bd4c58822942a7490e6', '8737': 'b5d365427b18431d3c962a8184d620176cf0c8541469720cbc9aca5b892330a6', '8759': '4ef9d485257a7e052b8731e0e7cecd2d1acfd13f5d4e9f3ed9131f5cbd2dd15f', '8760': '83f53b49c08848bcc25b31738ac9d1a48cfb9aa0bf44bef277d40226f0a233c5', '8765': 'f9678ba9e852139905d532f36764bf01da9442302322dc859fdec0413762e855', '8768': '7d29643a30e4c529b7108410fae7c8854a86d3d0751b9597ea4a050087287c92', '8769': '83f53b49c08848bcc25b31738ac9d1a48cfb9aa0bf44bef277d40226f0a233c5', '8774': '3dcb43a7d32fa6f1c227100d15b315db8519fcef7878df9dee34f79c39d84d1d', '8848': 'b3a9e32c904d728719eab27656cae4a3c769c2b5d2424f45c15cedc3a4fe6fb2', '8876': '174360acea5bfdfb12c1c73c88e5609360b0ba2278c5dc5e293f3c7d8401bed2', '8877': '142758aff462348a4a4b9a8d6afb4e61c4ac4b6a71ccdbce6855192ce0845895', '8878': '1297c63eae1002008ad50849f13685208906448bd4157b56facb3e9b84130ec3', '8882': 'f0ff3a33e71cba52b0192fbc1a682602a12b8f80aff7c82e8da4c125d4effc4c', '8889': 'a703e515c47e52dcd144c15989b27d336ba9825e161e638537d80991851a1ea4', '8959': '93bc88d036a697d382446544594c15b2742ab90c25e44517a263621004aeb447', '8961': '1378391e428ef643a91b162c963b76bfc8a02627640f4694ae359e243fd579ba', '8985': '9e60b030e3fc3416bad089537b6e1b323d5ef6d06e9f66546b63d156fd4ffa45', '8990': '3dcb43a7d32fa6f1c227100d15b315db8519fcef7878df9dee34f79c39d84d1d'}
saved = pd.read_csv(io.StringIO(SAVED_SAMPLE_CSV))
assert len(saved) == 500 and saved.idx.nunique() == 500
meta = ds.remove_columns('image').to_pandas()
meta['content_type'] = meta.metadata.apply(lambda m: m['content_type'])
meta['anatomy'] = meta.metadata.apply(lambda m: m['location'])
meta['question_id'] = meta.metadata.apply(lambda m: m['question_id'])
meta['idx'] = np.arange(len(meta))
GROUPS = {
    ('closed', 'anatomy'): 'anatomy_yesno',
    ('closed', 'pathology'): 'abnormality_yesno',
    ('closed', 'pathology_specific'): 'pathology_yesno',
    ('multiple_choice', 'anatomy'): 'anatomy_mcq',
    ('multiple_choice', 'pathology'): 'pathology_mcq',
}
meta['group'] = [GROUPS.get((q, c)) for q, c in zip(meta.question_type, meta.content_type)]
pool = meta[meta.group.notna()].copy()
pool['answer'] = pool.answer.str.strip()
pool.loc[pool.question_type == 'closed', 'answer'] = pool.loc[pool.question_type == 'closed', 'answer'].str.lower()
assert len(pool) == 7000
closed = pool[pool.question_type == 'closed']
mcq = pool[pool.question_type == 'multiple_choice']
parts = [g.groupby('answer').sample(50, random_state=SEED) for _, g in closed.groupby('group')]
parts += [g.sample(100, random_state=SEED) for _, g in mcq.groupby('group')]
resampled = pd.concat(parts).sort_values('idx').reset_index(drop=True)
columns = ['idx', 'question_id', 'group']
seed_match = resampled[columns].astype(str).equals(saved[columns].astype(str))
print('Seed-42 sample exactly matches saved sample:', seed_match)

# Saved indices identify the actual experiment even if library sampling behaviour changes.
selected = pool.set_index('idx').loc[saved.idx].reset_index()
assert selected[columns].astype(str).equals(saved[columns].astype(str)), 'Saved metadata does not match dataset rows.'
question_match = all(hashlib.sha256(str(r.question).strip().encode()).hexdigest() == QUESTION_SHA256[str(r.idx)] for r in selected.itertuples())
assert question_match, 'Question fingerprints differ. Recover the original dataset revision before reporting counts.'
print('All 500 question fingerprints verified against original predictions.')
if not seed_match:
    print('Sampling differs in this environment. Counts below use the verified ORIGINAL saved sample.')


In [ ]:
def column_values(field):
    if field.startswith('metadata.'):
        key = field.split('.', 1)[1]
        return selected.metadata.apply(lambda m: m.get(key))
    if field in selected.columns:
        return selected[field]
    raise ValueError(f'Unknown image ID field: {field}')

candidates = [c for c in ['image_id', 'image_uid', 'image_path', 'image_filename'] if c in selected.columns]
metadata_keys = set().union(*(m.keys() for m in selected.metadata))
candidates += ['metadata.' + c for c in ['image_id', 'image_uid', 'image_path', 'image_filename'] if c in metadata_keys]
print('Candidate image identifier fields:', candidates)
if IMAGE_ID_FIELD is None and candidates:
    IMAGE_ID_FIELD = candidates[0]
if IMAGE_ID_FIELD:
    values = column_values(IMAGE_ID_FIELD)
    assert values.notna().all() and values.map(lambda x: isinstance(x, (str, int)) and str(x) != '').all(), 'Image IDs must be nonempty scalar values.'
    selected['image_identifier'] = values.astype(str)
    print('Image identifier field:', IMAGE_ID_FIELD)
else:
    print('No explicit image identifier found. The report will give distinct decoded RGB image contents.')

# Pixel hashing does not assume that adjacent questions refer to the same image.
def image_digest(image):
    rgb = image.convert('RGB')
    header = json.dumps({'mode': 'RGB', 'size': rgb.size}, sort_keys=True).encode()
    return hashlib.sha256(header + b'\0' + rgb.tobytes()).hexdigest()

selected['pixel_hash'] = [image_digest(ds[int(i)]['image']) for i in tqdm(selected.idx, desc='Hashing 500 sampled images')]
if IMAGE_ID_FIELD:
    conflicts = selected.groupby('image_identifier').pixel_hash.nunique()
    assert conflicts.max() == 1, 'One image identifier maps to different pixel contents. Investigate before reporting.'

records = []
for name, group in [('overall', selected), *list(selected.groupby('group'))]:
    row = {'group': name, 'questions': len(group), 'unique_image_contents': group.pixel_hash.nunique()}
    if IMAGE_ID_FIELD:
        row['unique_image_identifiers'] = group.image_identifier.nunique()
    records.append(row)
counts = pd.DataFrame(records)
display(counts)
unit = 'image_identifier' if IMAGE_ID_FIELD else 'pixel_hash'
multiplicity = selected.groupby(unit).size().value_counts().sort_index()
repetitions = pd.DataFrame({'questions_per_image': multiplicity.index, 'number_of_images': multiplicity.values})
display(repetitions)
print('Group counts can overlap because one image can appear in multiple question groups.')


In [ ]:
OUTPUT.mkdir(parents=True, exist_ok=True)
counts.to_csv(OUTPUT / 'unique_image_counts.csv', index=False)
repetitions.to_csv(OUTPUT / 'questions_per_image.csv', index=False)
provenance = {
    'dataset': DATASET, 'configuration': 'benchmark', 'split': 'test',
    'resolved_dataset_commit': revision, 'seed': SEED,
    'saved_sample_sha256': hashlib.sha256(SAVED_SAMPLE_CSV.encode()).hexdigest(),
    'seed_sample_matches_saved': bool(seed_match), 'all_question_fingerprints_match': bool(question_match),
    'image_identifier_field': IMAGE_ID_FIELD,
    'image_content_definition': 'SHA-256 of decoded RGB dimensions and pixels',
    'python': platform.python_version(),
    'packages': {p: importlib.metadata.version(p) for p in ['datasets', 'pandas', 'numpy', 'pillow', 'huggingface_hub']},
    'counts': counts.to_dict(orient='records'),
}
(OUTPUT / 'audit_provenance.json').write_text(json.dumps(provenance, indent=2))
n = selected[unit].nunique()
if IMAGE_ID_FIELD:
    sentence = f'The 500 RadImageNet-VQA evaluation questions represent {n} unique dataset image identifiers, with {selected.pixel_hash.nunique()} distinct decoded RGB image contents.'
else:
    sentence = f'The 500 RadImageNet-VQA evaluation questions contain {n} distinct decoded RGB images, identified by image-content hashes. An explicit dataset image identifier was unavailable.'
report = sentence + '\nThe original saved sample was verified against all 500 question fingerprints.\nSeed-42 resampling exactly matched the saved sample: ' + str(seed_match) + '.\n'
(OUTPUT / 'reviewer_response_counts.txt').write_text(report)
print(report)
print('Saved reports to', OUTPUT)
